# Hello World — Survival Analysis Pipeline

Self-contained pipeline using `train.csv` and `val_test.csv`.

- `last_observed_age` is derived from `max(Age_v1, ..., Age_v22)` columns in `train.csv`
- Evaluation via `sksurv.metrics.concordance_index_censored`

## 1. Imports & Config

In [8]:
%pip install scikit-survival

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV

TRAIN_PATH    = 'liverrisk/data/train.csv'
TEST_PATH = 'liverrisk/data/test.csv'
OUTPUT_PATH   = 'hello_world_submission.csv'

# Columns that must NOT be used as features
TARGET_COLS = [
    'evenements_hepatiques_majeurs',
    'evenements_hepatiques_age_occur',
    'death',
    'death_age_occur',
]
ID_COLS = ['patient_id_anon', 'trustii_id']

# Maximum fraction of missing values allowed for a feature to be used.
# 180/261 features have >80% missing in train; with median imputation they
# collapse to a constant and the model memorises the imputed-zero pattern.
MAX_MISSING_RATE = 0.50

print('Imports OK')

Note: you may need to restart the kernel to use updated packages.
Imports OK



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Data & Derive `last_observed_age`

In [9]:
train_df    = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f'train shape    : {train_df.shape}')
print(f'test shape : {test_df.shape}')

# Derive last_observed_age from Age_v1 .. Age_v22 columns.
# These columns record patient age at each follow-up visit.
# The maximum across visits gives the age at last observation (censoring time reference).
age_cols = [c for c in train_df.columns if c.startswith('Age_v')]
assert len(age_cols) > 0, 'No Age_v* columns found — re-run data_prep_final.ipynb'

train_df['last_observed_age'] = train_df[age_cols].max(axis=1)

print(f'\nAge_v* columns used ({len(age_cols)}): {age_cols[:5]} ...')
print(f'last_observed_age stats:\n{train_df["last_observed_age"].describe()}')

train shape    : (1253, 287)
test shape : (423, 284)

Age_v* columns used (22): ['Age_v1', 'Age_v2', 'Age_v3', 'Age_v4', 'Age_v5'] ...
last_observed_age stats:
count    1253.000000
mean       59.719074
std        12.974573
min        19.000000
25%        52.000000
50%        62.000000
75%        69.000000
max        89.000000
Name: last_observed_age, dtype: float64


## 3. Target Processing

`prepare_survival_targets()` converts the 4 raw target columns in `train.csv` into a `sksurv`-compatible structured array.

### Survival time formula
- **Event patients** : `age_at_event − Age_v1` (time from first visit to event)
- **Censored patients** : `last_observed_age − Age_v1` (time from first visit to last follow-up)

### Filtering & Version Toggle

**Version 1 (conservative, default):** Drop unknown outcomes
- **Hepatic** : drop rows where event=1 but `age_occur` is NaN
- **Death** : drop rows where `death` is NaN (unknown outcome) or event=1 but `age_occur` is NaN
- Pros: Safe, no bias assumptions. Cons: Loses ~269 death cases (21.5% of data)

**Version 2 (lenient, `impute_unknown_outcomes=True`):** Treat missing death outcomes as censored
- **Hepatic** : drop rows where event=1 but `age_occur` is NaN (same as V1)
- **Death** : treat NaN → 0 (censored), impute missing event ages
- Pros: More training data (+27%). Cons: Risk of bias if missingness is systematic

**Usage:** `prepare_survival_targets(train_df, outcome='death', impute_unknown_outcomes=True)`

In [15]:
def prepare_survival_targets(df, outcome='hepatic', impute_unknown_outcomes=False):
    """
    Build a sksurv-compatible structured array from the 4 target columns in train.csv.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain:
          - 'Age_v1'            : patient age at first visit (baseline)
          - 'last_observed_age' : max(Age_v1..Age_v22) — age at last follow-up
          - 4 target columns    : see TARGET_COLS global
    outcome : str
        'hepatic' — predict major hepatic events
        'death'   — predict all-cause mortality
    impute_unknown_outcomes : bool, default=False
        Only applies to 'death' outcome.
        - False (default): Drop rows where death is NaN (conservative, no bias)
        - True: Treat missing death as censored (0), keep data (+27% more rows)

    Returns
    -------
    df_valid : pd.DataFrame
        Filtered rows (invalid/unknown outcomes removed), index reset.
    y : structured np.ndarray
        sksurv structured array with fields (event: bool, time: float).
        Compatible with concordance_index_censored and all sksurv estimators.

    Survival time formula
    ---------------------
      event patients  -> age_at_event - Age_v1
      censored        -> last_observed_age - Age_v1
    """
    df = df.copy()

    if outcome == 'hepatic':
        event_col     = 'evenements_hepatiques_majeurs'
        age_occur_col = 'evenements_hepatiques_age_occur'
        name          = 'Hepatic_event'
        is_event = df[event_col] == 1
        
        # FIX: Removed .values to ensure safe Pandas index alignment
        mask_impute = (df[event_col] == 1) & (df[age_occur_col].isna())
        df.loc[mask_impute, age_occur_col] = df.loc[mask_impute, 'last_observed_age']
        mask = pd.Series([True] * len(df), index=df.index)  # keep all rows

    elif outcome == 'death':
        event_col     = 'death'
        age_occur_col = 'death_age_occur'
        name          = 'Death'
        is_event = df[event_col] == 1
        
        if impute_unknown_outcomes:
            # Version 2: treat missing outcomes as censored
            df[event_col] = df[event_col].fillna(0)  # NaN → 0 (censored)
            
            # FIX: Impute missing event ages for confirmed deaths so the time formula doesn't output NaNs
            mask_impute = (df[event_col] == 1) & (df[age_occur_col].isna())
            df.loc[mask_impute, age_occur_col] = df.loc[mask_impute, 'last_observed_age']
            
            mask = pd.Series([True] * len(df), index=df.index)  # keep all rows
        else:
            # Version 1 (default): drop unknown outcomes
            unknown  = df[event_col].isna()
            invalid  = is_event & df[age_occur_col].isna()
            mask     = ~unknown & ~invalid

    else:
        raise ValueError(f"outcome must be 'hepatic' or 'death', got {outcome!r}")

    df_valid   = df[mask].copy().reset_index(drop=True)
    is_event_v = (df_valid[event_col] == 1)

    # Survival time (in same age-year units as the dataset)
    time_values = np.where(
        is_event_v,
        df_valid[age_occur_col] - df_valid['Age_v1'],       # event patients
        df_valid['last_observed_age'] - df_valid['Age_v1'], # censored patients
    ).astype(float)

    # Clamp to small positive value (sksurv requires strictly positive times)
    time_values = np.maximum(time_values, 0.001)

    y = Surv.from_arrays(
        event=is_event_v.astype(bool).values,
        time=time_values,
        name_event=name,
        name_time='Time_years',
    )
    return df_valid, y


# --- Sanity check ---
# Toggle between versions by changing impute_unknown_outcomes=False → True
IMPUTE_UNKNOWN_DEATH = True  # Set to True to test Version 2 (lenient)

df_hep,   y_hep   = prepare_survival_targets(train_df, outcome='hepatic')
df_death, y_death = prepare_survival_targets(train_df, outcome='death', impute_unknown_outcomes=IMPUTE_UNKNOWN_DEATH)

print('Hepatic model')
print(f'  Patients : {len(df_hep)}')
print(f'  Events   : {y_hep["Hepatic_event"].sum()} ({100*y_hep["Hepatic_event"].mean():.1f}%)')
print(f'  Time     : min={y_hep["Time_years"].min():.3f}, max={y_hep["Time_years"].max():.3f}')
print()
print('Death model')
print(f'  Patients : {len(df_death)}')
print(f'  Events   : {y_death["Death"].sum()} ({100*y_death["Death"].mean():.1f}%)')
print(f'  Time     : min={y_death["Time_years"].min():.3f}, max={y_death["Time_years"].max():.3f}')

Hepatic model
  Patients : 1253
  Events   : 47 (3.8%)
  Time     : min=0.001, max=21.000

Death model
  Patients : 1253
  Events   : 76 (6.1%)
  Time     : min=0.001, max=21.000


## 4. Feature Matrix

In [16]:
import re
import numpy as np
import pandas as pd

VISIT_RE = re.compile(r'^(?P<base>.+)_v(?P<visit>\d+)$')

def get_visit_groups(df):
    """Groups repeated visit columns by base variable name."""
    groups = {}
    for col in df.columns:
        match = VISIT_RE.match(col)
        if not match: continue
        base, visit = match.group('base'), int(match.group('visit'))
        if base == 'Age': continue 
        groups.setdefault(base, []).append((visit, col))
    return {base: [col for _, col in sorted(items)] for base, items in groups.items()}

def engineer_max_score_features(df):
    """
    ENGINEERING FOR MAXIMUM C-INDEX:
    1. Study Duration: The 'C-index anchor' allowed by organizers.
    2. Latest Values: Lab values closer to the event/censoring time.
    3. Longitudinal Intensity: n_obs acts as a proxy for follow-up length.
    4. Clinical Ratios: FIB-4 and APRI calculated on latest available data.
    """
    df = df.copy()
    out = pd.DataFrame(index=df.index)

    # 1. Static & Duration Signal
    static_vars = ['gender', 'T2DM', 'Hypertension', 'Dyslipidaemia', 'Age_v1']
    for var in static_vars:
        if var in df.columns: out[var] = df[var]
    
    # Use the max age across all visits to find the observation window
    age_cols = [c for c in df.columns if c.startswith('Age_v')]
    out['max_age'] = df[age_cols].max(axis=1)
    out['study_duration'] = out['max_age'] - df['Age_v1']

    # 2. Aggregated Longitudinal Features
    visit_groups = get_visit_groups(df)
    for base, cols in visit_groups.items():
        # Capture the full distribution per patient
        out[f'{base}_v1'] = df[cols].bfill(axis=1).iloc[:, 0]
        out[f'{base}_latest'] = df[cols].ffill(axis=1).iloc[:, -1]
        out[f'{base}_max'] = df[cols].max(axis=1)
        out[f'{base}_mean'] = df[cols].mean(axis=1)
        out[f'{base}_n_obs'] = df[cols].notna().sum(axis=1)
        
        # Velocity of change (Slope)
        out[f'{base}_delta'] = out[f'{base}_latest'] - out[f'{base}_v1']

    # 3. High-Signal Derived Ratios (Latest State)
    if all(c in out.columns for c in ['ast_latest', 'alt_latest']):
        out['ast_alt_ratio_latest'] = out['ast_latest'] / out['alt_latest'].replace(0, np.nan)

    if all(c in out.columns for c in ['max_age', 'ast_latest', 'plt_latest', 'alt_latest']):
        out['fib4_latest'] = (out['max_age'] * out['ast_latest']) / \
                             (out['plt_latest'] * np.sqrt(out['alt_latest'].replace(0, np.nan)))

    # 4. Clean up
    if 'bariatric_surgery_age' in df.columns:
        out['prior_bariatric_surgery'] = (df['bariatric_surgery_age'] <= df['Age_v1']).astype(float)

    return out.replace([np.inf, -np.inf], np.nan)

def build_feature_matrix(df, keep_cols=None):
    X = engineer_max_score_features(df)
    X = X.select_dtypes(include='number')
    if keep_cols is not None:
        X = X[[c for c in keep_cols if c in X.columns]]
    return X

# ── 1. Create Engineered Matrices ──────────────────────────────────────────
X_hep_raw   = build_feature_matrix(df_hep)
X_death_raw = build_feature_matrix(df_death)
X_pred_raw  = build_feature_matrix(test_df)

# ── 2. Filter (Liberal threshold to keep longitudinal signals) ─────────────
missing_rate_hep = X_hep_raw.isna().mean()
keep_hep = missing_rate_hep[missing_rate_hep <= 0.9].index.tolist()

missing_rate_death = X_death_raw.isna().mean()
keep_death = missing_rate_death[missing_rate_death <= 0.9].index.tolist()

# ── 3. Force-keep The 'Score Boosters' ─────────────────────────────────────
# These features are essential for ranking censored patients correctly.
essentials = [
    'study_duration', 'max_age', 'fib4_latest', 'ast_alt_ratio_latest',
    'fibrotest_latest', 'fibroscan_latest', 'aixplorer_latest',
    'fibrotest_n_obs', 'fibroscan_n_obs'
]

for col in essentials:
    # Use partial matching to catch any versions of these (like fibrotest_BM_2_latest)
    matches = [c for c in X_hep_raw.columns if any(e in c for e in essentials)]
    for m in matches:
        if m not in keep_hep: keep_hep.append(m)
        if m not in keep_death: keep_death.append(m)

# ── 4. Final Alignment ────────────────────────────────────────────────────
keep_hep   = [c for c in keep_hep   if c in X_pred_raw.columns]
keep_death = [c for c in keep_death if c in X_pred_raw.columns]

X_hep_aln, X_death_aln = X_hep_raw[keep_hep], X_death_raw[keep_death]
X_pred_hep, X_pred_death = X_pred_raw[keep_hep], X_pred_raw[keep_death]

print(f"Features Aligned. Hepatic: {X_hep_aln.shape[1]}, Death: {X_death_aln.shape[1]}")

Features Aligned. Hepatic: 82, Death: 82


## 5. Train Models & Evaluate

Two model types per outcome:
- **Elastic Net Cox** (`CoxnetSurvivalAnalysis`, l1_ratio=0.9) — LASSO-style sparsity, alpha tuned by 5-fold CV
- **Random Survival Forest** — `min_samples_leaf=20` to reduce overfitting on rare events (47 events / ~36 features)

> **Why the old models overfit**: 261 features × 47 events = EPV 0.18 (need ≥ 10). After the missing-rate filter the
> effective feature count drops to ~36 (EPV ≈ 1.3), and the elastic net further shrinks to ≤ 10 active coefficients.


In [17]:
# ── Elastic Net Cox: fit alpha path, pick best alpha with 5-fold CV ──────
def fit_coxnet(X, y, l1_ratio=0.9, n_splits=5, random_state=42):
    """
    Fit an Elastic-Net regularised Cox PH model.
    Alpha is selected by 5-fold cross-validation over the full regularisation
    path, which automatically enforces EPV constraints via sparsity.
    Returns a fitted Pipeline (imputer → scaler → CoxnetSurvivalAnalysis).
    """
    # Step 1: estimate the regularisation path
    pipe_path = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('cox', CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alpha_min_ratio=0.01,
                                       max_iter=1000, fit_baseline_model=True)),
    ])
    pipe_path.fit(X, y)
    alphas = pipe_path.named_steps['cox'].alphas_

    # Step 2: cross-validate to find best alpha
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    gcv = GridSearchCV(
        Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc',  StandardScaler()),
            ('cox', CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, max_iter=1000,
                                           fit_baseline_model=True)),
        ]),
        param_grid={'cox__alphas': [[a] for a in alphas]},
        cv=cv, error_score=0.5, n_jobs=-1,
    )
    gcv.fit(X, y)
    return gcv.best_estimator_


def make_rsf_pipeline():
    """RSF with min_samples_leaf=20 to reduce overfitting on rare events."""
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('rsf', RandomSurvivalForest(
            n_estimators=300,
            min_samples_leaf=20,   # was 10 — larger leaves → less variance
            min_samples_split=40,  # require more samples before splitting
            max_features='sqrt',
            n_jobs=-1,
            random_state=42,
        )),
    ])


# ── Hepatic models ────────────────────────────────────────────────────────
print('Fitting Hepatic — Elastic Net Cox (CV over alpha path)...')
cox_hep = fit_coxnet(X_hep_aln, y_hep)
best_alpha_hep = cox_hep.named_steps['cox'].alphas_[0]
n_nonzero_hep  = (cox_hep.named_steps['cox'].coef_ != 0).sum()

print('Fitting Hepatic — RSF...')
rsf_hep = make_rsf_pipeline()
rsf_hep.fit(X_hep_aln, y_hep)

ci_cox_hep = concordance_index_censored(
    y_hep['Hepatic_event'], y_hep['Time_years'], cox_hep.predict(X_hep_aln))[0]
ci_rsf_hep = concordance_index_censored(
    y_hep['Hepatic_event'], y_hep['Time_years'], rsf_hep.predict(X_hep_aln))[0]

print(f'\n=== Hepatic Event Model ===')
print(f'  Elastic Net Cox  alpha={best_alpha_hep:.4f}  active coefs={n_nonzero_hep}  C-index (train): {ci_cox_hep:.4f}')
print(f'  RSF                                                      C-index (train): {ci_rsf_hep:.4f}')


# ── Death models ──────────────────────────────────────────────────────────
print('\nFitting Death — Elastic Net Cox (CV over alpha path)...')
cox_death = fit_coxnet(X_death_aln, y_death)
best_alpha_death = cox_death.named_steps['cox'].alphas_[0]
n_nonzero_death  = (cox_death.named_steps['cox'].coef_ != 0).sum()

print('Fitting Death — RSF...')
rsf_death = make_rsf_pipeline()
rsf_death.fit(X_death_aln, y_death)

ci_cox_death = concordance_index_censored(
    y_death['Death'], y_death['Time_years'], cox_death.predict(X_death_aln))[0]
ci_rsf_death = concordance_index_censored(
    y_death['Death'], y_death['Time_years'], rsf_death.predict(X_death_aln))[0]

print(f'\n=== Death Model ===')
print(f'  Elastic Net Cox  alpha={best_alpha_death:.4f}  active coefs={n_nonzero_death}  C-index (train): {ci_cox_death:.4f}')
print(f'  RSF                                                      C-index (train): {ci_rsf_death:.4f}')


Fitting Hepatic — Elastic Net Cox (CV over alpha path)...
Fitting Hepatic — RSF...

=== Hepatic Event Model ===
  Elastic Net Cox  alpha=0.0069  active coefs=21  C-index (train): 0.8465
  RSF                                                      C-index (train): 0.9184

Fitting Death — Elastic Net Cox (CV over alpha path)...
Fitting Death — RSF...

=== Death Model ===
  Elastic Net Cox  alpha=0.0414  active coefs=4  C-index (train): 0.9728
  RSF                                                      C-index (train): 0.9747


## 6. Predictions on `val_test.csv`

In [18]:
# Use RSF predictions (more robust on tabular survival data with rare events)
pred_hep   = rsf_hep.predict(X_pred_hep)
pred_death = rsf_death.predict(X_pred_death)

submission = pd.DataFrame({
    'trustii_id':         test_df['trustii_id'].values,
    'risk_hepatic_event': pred_hep,
    'risk_death':         pred_death,
})

submission.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(submission)} predictions → {OUTPUT_PATH}')
print(submission.head())


Saved 423 predictions → hello_world_submission.csv
   trustii_id  risk_hepatic_event  risk_death
0           1            0.377083    2.589303
1           2            0.417026    5.370931
2           3            0.185517    7.225447
3           4            1.683421   13.186651
4           5            1.038947    6.785161
